In [ ]:
from datetime import timezone
import pandas as pd

from engine.technical import bullish_engulf
from engine.candles_loader import CandlesLoader
from engine.pattern_engine import PatternEngine
from engine.pattern_evaluator import PatternEvaluator

In [ ]:
def date_to_epoch(date_time_str, t_zone=timezone.utc):
    dt = pd.to_datetime(date_time_str, format="%d/%m/%Y %H:%M:%S")
    dt_utc = dt.tz_localize(t_zone)
    return int(dt_utc.timestamp())


def p_func(candles):
    return bullish_engulf(candles)


def inv_val(candles, current_candle):
    return candles.at[current_candle - 1, 'LOW']


def inv_cond(pattern, candles, current_candle):
    return candles.at[current_candle, 'LOW'] <= pattern.invalidation_price


def is_suc(pattern, price_change):
    return price_change > 0

In [ ]:
loader = CandlesLoader("../data/BTCUSD_1d_coinbase_of.csv")

In [ ]:
evaluator = PatternEvaluator(
    is_pattern=p_func,
    get_invalidation_val=inv_val,
    is_invalidated=inv_cond,
    is_successful=is_suc
)

In [ ]:
epoch_intervals = [(date_to_epoch("15/12/2018 00:00:00"), date_to_epoch("30/10/2021 00:00:00"))]
epoch_titles = ["Bull 2019-2021"]
sessions_open_list = list(range(1, 8))

In [ ]:
engine = PatternEngine(
    loader=loader,
    evaluator=evaluator,
    intervals=epoch_intervals,
    interval_labels=epoch_titles,
    sessions_open_list=sessions_open_list
)

In [ ]:
engine.run(show_results=True, plot_patterns=False)

In [ ]:
engine.run_by_regime(target_regimes=['BULLISH', 'BEARISH', 'RANGING'])

In [ ]:
results_filename = "results_datacube.parquet"
engine.save_results(results_filename)

In [ ]:
engine.show_stats_per_session(regime='ALL', print_stats=True, plot_histogram=False)

In [ ]:
engine.show_stats_progress_in_intervals(regime='ALL', mode="both")

In [ ]:
engine.plot_heatmap(metric='win_rate', regime='ALL')

In [ ]:
engine.plot_decay(metric='win_rate')

In [ ]:
engine.plot_3d(metric='sharpe_ratio')

In [ ]:
engine.plot_risk_reward(use_atr=True)

In [ ]:
engine.show_tactical_table(session_open=5, regime='ALL')